# Forecasting Railway Passenger Volumes (2026-2027)

This notebook estimates railway passenger volumes for the months that are not
yet available in the BPS data — **September 2026 through December 2027** — with
prediction intervals, accuracy evaluation, and a growth analysis.

The approach in short:

- **Training window:** 2022-2026 (post-COVID regime).
- **Categories:** all nine categories in `data/combined_data.csv`.
- **Candidate models:** Seasonal Naive, ETS (Holt-Winters), SARIMA, and
  HistGradientBoosting with seasonal lag features.
- **Model selection:** lowest backtest MAPE per category (expanding-origin,
  12-month horizon), so categories may use different models.
- **Units:** million passengers.

> Data source: BPS. `Total` and `Jawa (...)` are aggregate categories — never
> sum them together with their components (double counting).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATA_PATH = "data/combined_data.csv"
TRAIN_START, TRAIN_END = 2022, 2026
FORECAST_END = "2027-12-01"          # forecast up to Dec 2027
ANNUAL_PERIOD = 12
RANDOM_STATE = 42
Z80 = 1.2815515                      # normal quantile for an 80% interval
Z95 = 1.9599640                      # normal quantile for a 95% interval

BULAN_MAP = {
    1: "Januari", 2: "Februari", 3: "Maret", 4: "April", 5: "Mei",
    6: "Juni", 7: "Juli", 8: "Agustus", 9: "September", 10: "Oktober",
    11: "November", 12: "Desember",
}

FEATURE_COLS = ["month", "quarter", "lag1", "lag2", "lag3", "lag6", "lag12",
                "roll3", "roll12"]

# ----------------------------------------------------------------------
# Shared colour palette & Plotly theme (applied to every chart)
# ----------------------------------------------------------------------
C_HIST = "#94a3b8"       # historical line
C_FC = "#1f6feb"         # forecast line
C_FC2 = "#0b3d91"        # forecast accent
C_OBS = "#1f6feb"        # observed portion in bars
C_PART = "#f59e0b"       # forecast portion in bars
C_GROW = "#0ca678"       # growth line
C_BAND80 = "rgba(31,111,235,0.30)"
C_BAND95 = "rgba(31,111,235,0.14)"
C_DIV = "#f59e0b"        # "forecast starts" divider

pio.templates["kai"] = go.layout.Template(layout=go.Layout(
    font=dict(family="Segoe UI, Inter, Arial", size=13, color="#1e293b"),
    title=dict(font=dict(size=18, color="#0f172a"), x=0.02, xanchor="left",
               y=0.97, yanchor="top"),
    paper_bgcolor="white", plot_bgcolor="white",
    margin=dict(l=75, r=35, t=80, b=60),
    xaxis=dict(showgrid=False, linecolor="#e2e8f0", ticks="outside",
               tickcolor="#cbd5e1", title_font=dict(size=13)),
    yaxis=dict(gridcolor="#eef2f7", zeroline=False, linecolor="#e2e8f0",
               title_font=dict(size=13)),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
                bgcolor="rgba(0,0,0,0)", font=dict(size=12)),
    hoverlabel=dict(bgcolor="white", font_size=12, bordercolor="#cbd5e1",
                    font_family="Segoe UI, Inter, Arial"),
    colorway=["#1f6feb", "#e8590c", "#0ca678", "#7048e8", "#f59e0b"],
))
pio.templates.default = "kai"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"font.family": "Segoe UI", "axes.titleweight": "bold",
                     "axes.titlesize": 11, "axes.edgecolor": "#cbd5e1",
                     "figure.facecolor": "white"})

np.random.seed(RANDOM_STATE)
pd.options.display.float_format = "{:,.3f}".format
print("Setup complete. Training window:", TRAIN_START, "-", TRAIN_END)

## 1. Data Preparation

Load the combined dataset, normalise the unit to **million passengers**, and
identify the last month with real data so the forecast picks up exactly where
the observations end.

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw["Tanggal"] = pd.to_datetime(raw["Tanggal"])

# Normalise the values: "-" -> 0, then coerce to numeric.
raw["Jumlah Penumpang"] = pd.to_numeric(
    raw["Jumlah Penumpang"].astype(str).str.replace("-", "0"), errors="coerce"
).fillna(0)

# Original unit is thousands of passengers -> convert to millions.
raw["Penumpang (Juta)"] = raw["Jumlah Penumpang"] / 1000

print("Data shape:", raw.shape)
print("Date range:", raw["Tanggal"].min().date(), "->", raw["Tanggal"].max().date())
print("Categories:", raw["Kategori"].nunique())
raw.head()

In [ ]:
# Drop trailing months that hold no real observations yet
# (e.g. Sep-Dec 2026 are recorded as 0).
total_rows = raw[raw["Kategori"] == "Total"]
last_actual = total_rows.loc[total_rows["Jumlah Penumpang"] > 0, "Tanggal"].max()
raw = raw[raw["Tanggal"] <= last_actual].copy()

months_per_year = raw.groupby("Tahun")["Bulan_num"].nunique()
partial_years = months_per_year[months_per_year < 12].index.tolist()

print("Last month with real data:", last_actual.date())
print("\nMonths observed per year:")
print(months_per_year.to_string())
print("\nPartial year(s) requiring care:", partial_years)
print("Forecast horizon begins:", (last_actual + pd.DateOffset(months=1)).date())

In [ ]:
def make_series(kategori, df=None):
    '''Monthly series (million passengers) for one category.'''
    d = raw if df is None else df
    s = (d[d["Kategori"] == kategori]
         .sort_values("Tanggal")
         .set_index("Tanggal")["Penumpang (Juta)"])
    return s.asfreq("MS")


def meaningful_history(kategori):
    '''Trim leading zeros so the series starts when the mode actually operates.'''
    s = make_series(kategori)
    nz = s[s > 0]
    if len(nz) == 0:
        return s.iloc[0:0]
    return s[s.index >= nz.index.min()]


def train_series(kategori):
    '''Training-window series (2022-2026) based on its meaningful history.'''
    s = meaningful_history(kategori)
    return s[(s.index.year >= TRAIN_START) & (s.index.year <= TRAIN_END)]


KATEGORI = list(raw["Kategori"].unique())

summary = pd.DataFrame({
    "Kategori": KATEGORI,
    "Bulan": [len(train_series(k)) for k in KATEGORI],
    "Mulai": [train_series(k).index.min().date() for k in KATEGORI],
    "Akhir": [train_series(k).index.max().date() for k in KATEGORI],
})
summary

## 2. Exploratory Time-Series Analysis

Inspect the trend and seasonality of the `Total` series (STL decomposition) and
test for stationarity to guide the SARIMA specification.

In [ ]:
full_total = make_series("Total")
train_total = train_series("Total")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=full_total.index, y=full_total.values, mode="lines",
    name="Observed (2016-2026)", line=dict(color=C_HIST, width=2)))
fig.add_trace(go.Scatter(
    x=train_total.index, y=train_total.values, mode="lines",
    name="Training window (2022-2026)",
    line=dict(color=C_FC, width=3),
    fill="tozeroy", fillcolor="rgba(31,111,235,0.06)"))

fig.update_layout(
    title="Total Railway Passengers — Observed vs Training Window",
    xaxis_title="Period", yaxis_title="Million Passengers",
    hovermode="x unified")
fig.show()

In [ ]:
stl = STL(train_total, period=ANNUAL_PERIOD, robust=True).fit()

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=("Observed", "Trend", "Seasonal", "Residual"))
series_defs = [
    (train_total.values, "Observed", C_FC2),
    (stl.trend, "Trend", "#e8590c"),
    (stl.seasonal, "Seasonal", "#0ca678"),
    (stl.resid, "Residual", C_HIST),
]
for i, (vals, name, color) in enumerate(series_defs, start=1):
    fig.add_trace(go.Scatter(x=train_total.index, y=vals, mode="lines",
                             name=name, line=dict(color=color, width=2)), i, 1)
fig.update_layout(height=760, showlegend=False,
                  title="STL Decomposition — Total Passengers (2022-2026)",
                  hovermode="x unified")
fig.show()

In [ ]:
adf = adfuller(train_total.dropna())
print(f"ADF statistic = {adf[0]:.3f} | p-value = {adf[1]:.4f}")
print("Conclusion:", "stationary series" if adf[1] < 0.05 else
      "non-stationary series (SARIMA uses differencing)")

var_resid = np.var(stl.resid)
strength_seasonal = max(0.0, 1 - var_resid / np.var(stl.seasonal + stl.resid))
strength_trend = max(0.0, 1 - var_resid / np.var(stl.trend + stl.resid))
print(f"Seasonal strength = {strength_seasonal:.3f}")
print(f"Trend strength    = {strength_trend:.3f}")

## 3. Backtesting and Model Selection

Four candidates are evaluated with **expanding-origin** backtesting: train on
the data before a test year, then forecast the following 12 months. Metrics are
MAPE, RMSE, and MAE; the best model per category is chosen by MAPE.

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    m = y_true != 0
    if m.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

In [ ]:
def make_ml_features(series):
    '''Build seasonal lag features for the ML model (target = current month).'''
    df = pd.DataFrame({"y": series.values}, index=series.index)
    df["month"] = df.index.month
    df["quarter"] = df.index.quarter
    for lag in (1, 2, 3, 6, 12):
        df[f"lag{lag}"] = df["y"].shift(lag)
    df["roll3"] = df["y"].shift(1).rolling(3).mean()
    df["roll12"] = df["y"].shift(1).rolling(12).mean()
    df = df.dropna()
    return {"X": df[FEATURE_COLS], "y": df["y"], "index": df.index}


def make_feature_row(series, date):
    '''Feature row for one future date (used in the recursive forecast).'''
    row = {"month": date.month, "quarter": date.quarter}
    for lag in (1, 2, 3, 6, 12):
        row[f"lag{lag}"] = series.iloc[-lag] if len(series) >= lag else np.nan
    row["roll3"] = series.iloc[-3:].mean()
    row["roll12"] = series.iloc[-12:].mean()
    return pd.DataFrame([row], columns=FEATURE_COLS)


def ml_forecast(model, series, h):
    '''Recursive h-step forecast with updated lag features.'''
    work = series.astype(float).copy()
    preds = []
    for _ in range(h):
        next_date = work.index[-1] + pd.DateOffset(months=1)
        X_next = make_feature_row(work, next_date)
        p = float(model.predict(X_next)[0])
        preds.append(p)
        work = pd.concat([work, pd.Series([p], index=pd.DatetimeIndex([next_date]))])
    return np.array(preds)

In [ ]:
def _clip_ci(mean, lo80, hi80, lo95, hi95):
    '''Keep forecasts and bounds non-negative and free of NaN.'''
    mean = np.nan_to_num(np.clip(np.asarray(mean, float), 0, None))
    lo80 = np.nan_to_num(np.clip(np.asarray(lo80, float), 0, None))
    hi80 = np.nan_to_num(np.clip(np.asarray(hi80, float), 0, None))
    lo95 = np.nan_to_num(np.clip(np.asarray(lo95, float), 0, None))
    hi95 = np.nan_to_num(np.clip(np.asarray(hi95, float), 0, None))
    return (mean, lo80, hi80, lo95, hi95)


def model_fit_predict(name, train, h):
    '''Fit one model and return (mean, lo80, hi80, lo95, hi95).

    Returns None when the model cannot be fitted (too little data, did not
    converge, or produced an implausible forecast).
    '''
    train = train.astype(float)
    try:
        if name == "SeasonalNaive":
            if len(train) < ANNUAL_PERIOD:
                return None
            vals = np.resize(train.values[-ANNUAL_PERIOD:], h).astype(float)
            resid = train.values[ANNUAL_PERIOD:] - train.values[:-ANNUAL_PERIOD]
            sigma = float(np.nanstd(resid)) if len(resid) else float(np.nanstd(train.values))
            return _clip_ci(vals, vals - Z80 * sigma, vals + Z80 * sigma,
                            vals - Z95 * sigma, vals + Z95 * sigma)

        if name == "ETS":
            # Use seasonality only when at least two full cycles are available.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                if len(train) >= 2 * ANNUAL_PERIOD:
                    model = ExponentialSmoothing(
                        train, trend="add", damped_trend=True, seasonal="add",
                        seasonal_periods=ANNUAL_PERIOD,
                        initialization_method="estimated").fit()
                else:
                    model = ExponentialSmoothing(
                        train, trend="add", damped_trend=True, seasonal=None,
                        initialization_method="estimated").fit()
            f = np.asarray(model.forecast(h), float)
            sigma = float(np.nanstd(model.resid))
            return _clip_ci(f, f - Z80 * sigma, f + Z80 * sigma,
                            f - Z95 * sigma, f + Z95 * sigma)

        if name == "SARIMA":
            # Pick the seasonal order with the best AIC; reject non-convergent fits.
            best = None
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                for sorder in [(1, 1, 1, ANNUAL_PERIOD), (0, 1, 1, ANNUAL_PERIOD),
                               (1, 0, 1, ANNUAL_PERIOD)]:
                    try:
                        res = SARIMAX(train, order=(1, 1, 1), seasonal_order=sorder,
                                      enforce_stationarity=False,
                                      enforce_invertibility=False).fit(disp=False)
                        if not res.mle_retvals.get("converged", True):
                            continue
                        if best is None or res.aic < best.aic:
                            best = res
                    except Exception:  # noqa: BLE001
                        continue
            if best is None:
                return None
            fc = best.get_forecast(h)
            mean = np.asarray(fc.predicted_mean, float)
            scale = np.nanmax(np.abs(train.values))
            if not np.all(np.isfinite(mean)) or np.nanmax(np.abs(mean)) > 5 * scale:
                return None
            ci80 = np.asarray(fc.conf_int(alpha=0.20), float)
            ci95 = np.asarray(fc.conf_int(alpha=0.05), float)
            return _clip_ci(mean, ci80[:, 0], ci80[:, 1], ci95[:, 0], ci95[:, 1])

        if name == "ML":
            if len(train) < 2 * ANNUAL_PERIOD:
                return None
            feat = make_ml_features(train)
            model = HistGradientBoostingRegressor(
                max_iter=300, learning_rate=0.05, max_depth=3,
                l2_regularization=1.0, random_state=RANDOM_STATE)
            model.fit(feat["X"], feat["y"])
            fitted = model.predict(feat["X"])
            sigma = float(np.std(feat["y"].values - fitted))
            vals = ml_forecast(model, train, h)
            return _clip_ci(vals, vals - Z80 * sigma, vals + Z80 * sigma,
                            vals - Z95 * sigma, vals + Z95 * sigma)
    except Exception:  # noqa: BLE001
        return None
    return None

In [ ]:
MODELS = ["SeasonalNaive", "ETS", "SARIMA", "ML"]


def backtest(kategori):
    '''Expanding-origin backtest: train before each test year, forecast ahead.'''
    s = train_series(kategori).astype(float)
    results = {}
    for name in MODELS:
        fold_metrics = []
        for test_year in (2024, 2025, 2026):
            test_mask = s.index.year == test_year
            if test_mask.sum() == 0:
                continue
            train = s[s.index.year < test_year]
            test = s[test_mask]
            if len(train) < ANNUAL_PERIOD:
                continue
            out = model_fit_predict(name, train, len(test))
            if out is None:
                continue
            pred = np.asarray(out[0], float)[:len(test)]
            fold_metrics.append({
                "tahun": test_year,
                "MAPE": mape(test.values, pred),
                "RMSE": rmse(test.values, pred),
                "MAE": mae(test.values, pred),
            })
        if fold_metrics:
            results[name] = {
                "MAPE": float(np.nanmean([f["MAPE"] for f in fold_metrics])),
                "RMSE": float(np.nanmean([f["RMSE"] for f in fold_metrics])),
                "MAE": float(np.nanmean([f["MAE"] for f in fold_metrics])),
                "n_fold": len(fold_metrics),
            }
    return results

In [ ]:
model_choice = {}
summary_rows = []
detail_records = []

for k in KATEGORI:
    res = backtest(k)
    if not res:
        print(f"{k:44s} -> no valid model; falling back to SeasonalNaive")
        model_choice[k] = "SeasonalNaive"
        summary_rows.append({"Kategori": k, "Model": "SeasonalNaive",
                             "MAPE": np.nan, "RMSE": np.nan, "MAE": np.nan,
                             "n_fold": 0})
        continue
    for name, m in res.items():
        detail_records.append({"Kategori": k, "Model": name,
                               "MAPE": m["MAPE"], "RMSE": m["RMSE"],
                               "MAE": m["MAE"], "n_fold": m["n_fold"]})
    best = min(res.items(), key=lambda kv: (np.isnan(kv[1]["MAPE"]), kv[1]["MAPE"]))
    model_choice[k] = best[0]
    summary_rows.append({"Kategori": k, "Model": best[0],
                         "MAPE": best[1]["MAPE"], "RMSE": best[1]["RMSE"],
                         "MAE": best[1]["MAE"], "n_fold": best[1]["n_fold"]})

backtest_summary = pd.DataFrame(summary_rows)
backtest_detail = pd.DataFrame(detail_records)

print("=== MAPE (%) per model per category ===")
print(backtest_detail.pivot(index="Kategori", columns="Model",
                            values="MAPE").round(2).to_string())
print("\n=== Selected model per category ===")
print(backtest_summary.round(3).to_string(index=False))

## 4. Final Training and Forecast (Sep 2026 - Dec 2027)

The best model for each category is refit on all 2022-2026 data and used to
forecast every month from the end of the observations up to December 2027
(16 months in total), with 80% and 95% intervals.

In [ ]:
def final_forecast(kategori, chosen_model):
    '''Forecast from the month after the last observation through FORECAST_END.'''
    s = train_series(kategori)
    start = s.index.max() + pd.DateOffset(months=1)
    future_idx = pd.date_range(start, FORECAST_END, freq="MS")
    h = len(future_idx)

    out = model_fit_predict(chosen_model, s, h)
    used = chosen_model
    if out is None:
        out = model_fit_predict("SeasonalNaive", s, h)
        used = f"{chosen_model}->SeasonalNaive"
    mean, lo80, hi80, lo95, hi95 = out
    return pd.DataFrame({
        "Kategori": kategori,
        "Tanggal": future_idx,
        "Ramalan": np.asarray(mean, float),
        "Batas_Bawah_80": np.asarray(lo80, float),
        "Batas_Atas_80": np.asarray(hi80, float),
        "Batas_Bawah_95": np.asarray(lo95, float),
        "Batas_Atas_95": np.asarray(hi95, float),
        "Model": used,
    })


fc_parts = []
for k in KATEGORI:
    fc = final_forecast(k, model_choice.get(k, "SeasonalNaive"))
    row = backtest_summary.loc[backtest_summary["Kategori"] == k, "MAPE"]
    fc["MAPE_uji"] = float(row.iloc[0]) if len(row) else np.nan
    fc_parts.append(fc)

forecast = pd.concat(fc_parts, ignore_index=True)
forecast["Tahun"] = forecast["Tanggal"].dt.year
forecast["Bulan_num"] = forecast["Tanggal"].dt.month
forecast["Bulan"] = forecast["Bulan_num"].map(BULAN_MAP)

for c in ("Batas_Bawah_80", "Batas_Atas_80", "Batas_Bawah_95", "Batas_Atas_95"):
    forecast[c] = forecast[c].clip(lower=0)

fc_total = forecast[forecast["Kategori"] == "Total"].sort_values("Tanggal")
print(f"Forecast horizon: {fc_total['Tanggal'].min().date()} -> "
      f"{fc_total['Tanggal'].max().date()} ({len(fc_total)} months)")
print(f"Total rows: {len(forecast)}")
fc_total[["Tanggal", "Ramalan", "Batas_Bawah_80", "Batas_Atas_80",
          "Batas_Bawah_95", "Batas_Atas_95", "Model"]].round(2)

In [ ]:
fc_total = forecast[forecast["Kategori"] == "Total"].sort_values("Tanggal")
part_2026 = fc_total[fc_total["Tahun"] == 2026]
full_2027 = fc_total[fc_total["Tahun"] == 2027]

print("=== Forecast summary: TOTAL passengers (million) ===")
print(f"Sep-Dec 2026 (remaining months)")
print(f"  Point       : {part_2026['Ramalan'].sum():>10,.2f}")
print(f"  Interval 80%: {part_2026['Batas_Bawah_80'].sum():>10,.2f}  to  {part_2026['Batas_Atas_80'].sum():,.2f}")
print()
print(f"Full year 2027")
print(f"  Point       : {full_2027['Ramalan'].sum():>10,.2f}")
print(f"  Interval 80%: {full_2027['Batas_Bawah_80'].sum():>10,.2f}  to  {full_2027['Batas_Atas_80'].sum():,.2f}")
print(f"  Interval 95%: {full_2027['Batas_Bawah_95'].sum():>10,.2f}  to  {full_2027['Batas_Atas_95'].sum():,.2f}")

annual = (forecast[forecast["Tahun"] == 2027]
          .groupby("Kategori")
          .agg(Ramalan=("Ramalan", "sum"),
               Lo80=("Batas_Bawah_80", "sum"),
               Hi80=("Batas_Atas_80", "sum"))
          .sort_values("Ramalan", ascending=False)
          .rename(columns={"Ramalan": "Total 2027", "Lo80": "80% low",
                           "Hi80": "80% high"}))
print("\n=== 2027 total per category (million) ===")
print(annual.round(2).to_string())

## 5. Growth Analysis and Summary

This section turns the forecast into an interpretable story: how the
**remaining months of 2026** (Sep-Dec), the **estimated full-year 2026**, and
**2027** compare, and where the growth comes from.

Definitions used below:

- **Observed 2026** = actual data, January-August 2026 only (the last real
  observations).
- **Estimated 2026 (full year)** = observed Jan-Aug 2026 **plus** forecast
  Sep-Dec 2026.
- **2027** = full-year forecast.
- **Growth** = percentage change of the estimated 2027 total against the
  estimated full-year 2026 total.

In [ ]:
# ----- Annual totals per category: observed and forecast -----
actual_kat = (raw.groupby(["Kategori", "Tahun"])["Penumpang (Juta)"]
              .sum().unstack("Tahun"))
fc_kat = (forecast.groupby(["Kategori", "Tahun"])["Ramalan"]
          .sum().unstack("Tahun"))

obs_2026 = actual_kat.get(2026, pd.Series(0.0, index=KATEGORI))     # Jan-Aug
fc_2026 = fc_kat.get(2026, pd.Series(0.0, index=KATEGORI))          # Sep-Dec
est_2026 = (obs_2026.add(fc_2026, fill_value=0)).reindex(KATEGORI).fillna(0)
fc_2027 = fc_kat.get(2027, pd.Series(0.0, index=KATEGORI)).reindex(KATEGORI).fillna(0)

growth_pct = np.where(est_2026 > 0, (fc_2027 / est_2026 - 1) * 100, np.nan)

growth_table = pd.DataFrame({
    "Observed 2026 (Jan-Aug)": obs_2026.reindex(KATEGORI).fillna(0),
    "Forecast 2026 (Sep-Dec)": fc_2026.reindex(KATEGORI).fillna(0),
    "Est. 2026 (full year)": est_2026,
    "Forecast 2027": fc_2027,
    "Growth 2026->2027 (%)": growth_pct,
}).sort_values("Forecast 2027", ascending=False)

print("=== Annual totals per category (million passengers) ===")
print(growth_table.round(2).to_string())

In [ ]:
# ----- Month-by-month year-over-year change for the Total series -----
obs_26 = make_series("Total")
obs_26 = obs_26[obs_26.index.year == 2026]                     # Jan-Aug actual
f_26 = fc_total[fc_total["Tahun"] == 2026].set_index("Tanggal")["Ramalan"]
month_2026 = pd.concat([obs_26, f_26])                         # full 2026

f_27 = fc_total[fc_total["Tahun"] == 2027].set_index("Tanggal")["Ramalan"]

# Align both years by month number (1-12) so the table has 12 clean rows.
m2026 = month_2026.groupby(month_2026.index.month).sum()
m2027 = f_27.groupby(f_27.index.month).sum()
monthly = pd.DataFrame({"2026": m2026, "2027": m2027})
monthly.index.name = "Bulan_num"
monthly["Bulan"] = monthly.index.map(BULAN_MAP)
monthly["MoM 2027 (%)"] = monthly["2027"].pct_change() * 100
monthly["YoY (%)"] = (monthly["2027"] / monthly["2026"] - 1) * 100
print("=== Total: 2027 monthly forecast vs 2026 (million) ===")
print(monthly.round(2).to_string())

# ----- Headline numbers for the narrative -----
obs_2026_total = obs_26.sum()
fc_2026_total = f_26.sum()
est_2026_total = obs_2026_total + fc_2026_total
est_2027_total = f_27.sum()
growth_2027 = (est_2027_total / est_2026_total - 1) * 100
lo80_2027 = fc_total[fc_total["Tahun"] == 2027]["Batas_Bawah_80"].sum()
hi80_2027 = fc_total[fc_total["Tahun"] == 2027]["Batas_Atas_80"].sum()

top = growth_table.drop(index="Total", errors="ignore").iloc[0]
fastest = growth_table.drop(index="Total", errors="ignore")[
    "Growth 2026->2027 (%)"].idxmax()
fastest_val = growth_table.loc[fastest, "Growth 2026->2027 (%)"]

print("\n================ SUMMARY (Total) ================")
print(f"Observed Jan-Aug 2026          : {obs_2026_total:,.2f} million")
print(f"Forecast Sep-Dec 2026          : {fc_2026_total:,.2f} million")
print(f"Estimated full-year 2026       : {est_2026_total:,.2f} million")
print(f"Forecast full-year 2027        : {est_2027_total:,.2f} million "
      f"(80%: {lo80_2027:,.2f} - {hi80_2027:,.2f})")
print(f"Expected growth 2026 -> 2027   : {growth_2027:+.2f}%")
print(f"Largest category in 2027       : {top.name} ({top['Forecast 2027']:,.2f} million)")
print(f"Fastest-growing category       : {fastest} ({fastest_val:+.2f}%)")
print("=================================================")

### How to read the summary

- **Observed Jan-Aug 2026** is the only real data point for the year, so it is
  used as the launch pad. **Sep-Dec 2026** is model output and completes the
  year estimate.
- The **estimated full-year 2026** is therefore a *blend* of observed and
  forecast months. Comparisons against 2027 inherit the uncertainty of the four
  forecast months, so treat the growth number as indicative.
- Because 2026 has fewer observed months, the year-over-year growth of the
  `Total` series is best read from the monthly chart (2027 vs 2026 by month)
  rather than from a single annual figure.
- For a cleaner comparison, the notebook also shows full-year 2025 (all
  observed) against the 2027 forecast in the growth charts.

## 6. Visualisations

Interactive, easy-to-read charts:

1. Headline chart — `Total` observed plus forecast with intervals.
2. Per-category panel — every category's observed history and forecast.
3. Annual totals 2022-2027 — bars split into observed vs forecast, with the
   year-over-year growth rate overlaid.
4. Growth 2026 to 2027 by category — who grows fastest.
5. Monthly forecast with month-over-month change for 2027.

In [ ]:
def add_forecast_bands(fig, sub, row=None, col=None):
    '''Add 95% and 80% bands for a forecast table.'''
    x = list(sub["Tanggal"])
    fig.add_trace(go.Scatter(
        x=x + x[::-1],
        y=list(sub["Batas_Atas_95"]) + list(sub["Batas_Bawah_95"])[::-1],
        fill="toself", fillcolor=C_BAND95, line=dict(color="rgba(0,0,0,0)"),
        name="95% interval", hoverinfo="skip", showlegend=row is None),
        row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x + x[::-1],
        y=list(sub["Batas_Atas_80"]) + list(sub["Batas_Bawah_80"])[::-1],
        fill="toself", fillcolor=C_BAND80, line=dict(color="rgba(0,0,0,0)"),
        name="80% interval", hoverinfo="skip", showlegend=row is None),
        row=row, col=col)


hist = make_series("Total")
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode="lines",
                         name="Observed", line=dict(color=C_HIST, width=2)))
add_forecast_bands(fig, fc_total)
fig.add_trace(go.Scatter(
    x=fc_total["Tanggal"], y=fc_total["Ramalan"], mode="lines+markers",
    name="Forecast", line=dict(color=C_FC, width=3, dash="dash"),
    marker=dict(size=6, line=dict(width=1, color="white")),
    hovertemplate="%{x|%b %Y}<br>Forecast: %{y:,.2f} M<extra></extra>"))
fig.add_vline(x=last_actual, line=dict(color=C_DIV, width=2, dash="dot"))
fig.add_annotation(x=last_actual, y=1.06, yref="paper", showarrow=False,
                   xanchor="left", font=dict(color=C_DIV, size=12),
                   text="forecast starts here")
fig.update_layout(
    title="Total Railway Passengers — Observed and Forecast to 2027",
    xaxis_title="Period", yaxis_title="Million Passengers",
    hovermode="x unified")
fig.show()

In [ ]:
non_total = [k for k in KATEGORI if k != "Total"]
fig = make_subplots(rows=3, cols=3, subplot_titles=non_total,
                    vertical_spacing=0.09, horizontal_spacing=0.06)

for i, k in enumerate(non_total):
    r, c = divmod(i, 3)
    hist_k = make_series(k)
    hist_k = hist_k[hist_k.index >= "2021-01-01"]     # one year of context
    sub = forecast[forecast["Kategori"] == k].sort_values("Tanggal")
    fig.add_trace(go.Scatter(x=hist_k.index, y=hist_k.values, mode="lines",
                             line=dict(color=C_HIST, width=1.6),
                             showlegend=False, name="Observed"), r + 1, c + 1)
    add_forecast_bands(fig, sub, row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(
        x=sub["Tanggal"], y=sub["Ramalan"], mode="lines+markers",
        line=dict(color=C_FC, width=2, dash="dash"),
        marker=dict(size=3), showlegend=False, name="Forecast"), r + 1, c + 1)
    fig.add_vline(x=last_actual, line=dict(color=C_DIV, width=1, dash="dot"),
                  row=r + 1, col=c + 1)

fig.update_layout(height=920, title="Forecast by Category — Sep 2026 to Dec 2027 "
                  "(80% interval shaded)", hovermode="x unified")
fig.update_annotations(font_size=12)
fig.show()

In [ ]:
# ----- Annual totals 2022-2027 (observed vs forecast) + growth rate -----
actual_annual = (raw[raw["Kategori"] == "Total"]
                 .groupby("Tahun")["Penumpang (Juta)"].sum())
years = [2022, 2023, 2024, 2025, 2026, 2027]
obs_part = [actual_annual.get(y, 0.0) for y in range(2022, 2027)] + [0.0]
fc_part = [0.0] * 4 + [fc_2026_total, est_2027_total]
totals = [o + f for o, f in zip(obs_part, fc_part)]
growth = [np.nan] + [(totals[i] / totals[i - 1] - 1) * 100
                     for i in range(1, len(totals))]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=[str(y) for y in years], y=obs_part, name="Observed",
    marker_color=C_OBS,
    hovertemplate="%{x}<br>Observed: %{y:,.1f} M<extra></extra>"))
fig.add_trace(go.Bar(
    x=[str(y) for y in years], y=fc_part, name="Forecast / estimated",
    marker_color=C_PART,
    hovertemplate="%{x}<br>Forecast: %{y:,.1f} M<extra></extra>"))
fig.add_trace(go.Scatter(
    x=[str(y) for y in years], y=totals, mode="lines+markers+text",
    name="Total", text=[f"{t:,.0f}" for t in totals], textposition="top center",
    line=dict(color=C_FC2, width=2), marker=dict(size=7),
    hovertemplate="%{x}<br>Total: %{y:,.1f} M<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(
    x=[str(y) for y in years], y=growth, mode="lines+markers",
    name="YoY growth (%)", line=dict(color=C_GROW, width=2, dash="dot"),
    marker=dict(size=6, symbol="diamond"),
    hovertemplate="%{x}<br>Growth: %{y:+.1f}%<extra></extra>"), secondary_y=True)

fig.update_layout(
    title="Annual Total Passengers 2022-2027 — Observed, Estimated, Forecast",
    barmode="stack", hovermode="x unified")
fig.update_yaxes(title_text="Million Passengers", secondary_y=False,
                 range=[0, max(totals) * 1.25])
fig.update_yaxes(title_text="YoY growth (%)", secondary_y=True, showgrid=False)
fig.show()

In [ ]:
# ----- Growth 2026 -> 2027 by category -----
gt = growth_table.drop(index="Total", errors="ignore").copy()
gt = gt.sort_values("Growth 2026->2027 (%)")
colors = ["#9be9c8" if v < 0 else "#0ca678" for v in gt["Growth 2026->2027 (%)"]]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=gt["Growth 2026->2027 (%)"], y=gt.index, orientation="h",
    marker_color=colors,
    text=[f"{v:+.1f}%" for v in gt["Growth 2026->2027 (%)"]],
    textposition="outside",
    hovertemplate="%{y}<br>Growth: %{x:+.1f}%<extra></extra>"))
fig.add_vline(x=0, line=dict(color="#94a3b8", width=1))
fig.update_layout(
    title="Passenger Growth 2026 to 2027 by Category (estimated full-year 2026 "
          "vs 2027 forecast)",
    xaxis_title="Growth (%)", yaxis_title="",
    margin=dict(l=220, r=80, t=80, b=60), showlegend=False)
fig.update_xaxes(range=[gt["Growth 2026->2027 (%)"].min() - 5,
                        gt["Growth 2026->2027 (%)"].max() + 6])
fig.show()

In [ ]:
# ----- Monthly forecast with month-over-month change for 2027 -----
m27 = fc_total[fc_total["Tahun"] == 2027].copy()
colors = ["#0ca678" if v >= 0 else "#e8590c" for v in m27["Ramalan"].diff().fillna(0)]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=m27["Tanggal"], y=m27["Ramalan"], name="Monthly forecast",
    marker_color=C_OBS, opacity=0.85,
    hovertemplate="%{x|%b %Y}<br>Forecast: %{y:,.2f} M<extra></extra>"))
fig.add_trace(go.Scatter(
    x=m27["Tanggal"], y=m27["Ramalan"].pct_change() * 100,
    mode="lines+markers", name="Month-over-month change (%)",
    line=dict(color=C_GROW, width=2), marker=dict(size=6),
    hovertemplate="%{x|%b %Y}<br>Change: %{y:+.1f}%<extra></extra>"),
    secondary_y=True)
fig.update_layout(
    title="2027 Monthly Total Forecast and Month-over-Month Change",
    xaxis_title="Month", hovermode="x unified")
fig.update_yaxes(title_text="Million Passengers", secondary_y=False,
                 range=[m27["Ramalan"].min() * 0.9, m27["Ramalan"].max() * 1.12])
fig.update_yaxes(title_text="MoM change (%)", secondary_y=True, showgrid=False)
fig.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=False)
plt.subplots_adjust(hspace=0.42, wspace=0.22)
for ax, k in zip(axes.flat, KATEGORI):
    hist_k = make_series(k)
    hist_k = hist_k[hist_k.index >= "2021-01-01"]
    sub = forecast[forecast["Kategori"] == k].sort_values("Tanggal")
    ax.plot(hist_k.index, hist_k.values, color=C_HIST, linewidth=1.6,
            label="Observed")
    ax.fill_between(sub["Tanggal"], sub["Batas_Bawah_95"], sub["Batas_Atas_95"],
                    color=C_FC, alpha=0.12, label="95% interval")
    ax.fill_between(sub["Tanggal"], sub["Batas_Bawah_80"], sub["Batas_Atas_80"],
                    color=C_FC, alpha=0.28, label="80% interval")
    ax.plot(sub["Tanggal"], sub["Ramalan"], color=C_FC, linestyle="--",
            linewidth=2, marker="o", markersize=3, label="Forecast")
    ax.axvline(last_actual, color=C_DIV, linestyle=":", linewidth=1.2)
    ax.set_title(k)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", labelsize=8)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
axes.flat[0].legend(fontsize=7, frameon=True)
fig.suptitle("Railway Passenger Forecast (Sep 2026 - Dec 2027) — million passengers",
             fontweight="bold", y=0.995)
fig.savefig("forecast_2027.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved chart: forecast_2027.png")

## 7. Save Results

Write the monthly forecasts (Sep 2026 - Dec 2027) to `data/forecast_2027.csv`.

In [ ]:
OUT_COLS = ["Kategori", "Tahun", "Bulan", "Bulan_num", "Tanggal", "Ramalan",
            "Batas_Bawah_80", "Batas_Atas_80", "Batas_Bawah_95",
            "Batas_Atas_95", "Model", "MAPE_uji"]

out = forecast.copy()
out["Tanggal"] = out["Tanggal"].dt.strftime("%Y-%m-%d")
out = out[OUT_COLS].sort_values(["Kategori", "Tanggal"])
out.to_csv("data/forecast_2027.csv", index=False)
print("Saved: data/forecast_2027.csv")
print("Rows:", len(out), "| Categories:", out["Kategori"].nunique(),
      "| Years:", sorted(out["Tahun"].unique()))
out.head(12)

## 8. Assumptions and Limitations

- **Training window 2022-2026** avoids the 2020-2021 COVID shock, which is a
  structural break. The trade-off is a short training history, so intervals can
  be wide.
- **2026 is partial** (observations stop in August); September-December 2026 are
  forecast, not observed. The full-year 2026 figure is therefore a blend of
  observed and forecast months and should not be treated as final data.
- **Short-lived categories** (Airport Railway, MRT, LRT, Whoosh, starting 2024)
  have few backtest folds, so their estimates are the least reliable.
- **Annual intervals** are obtained by summing monthly bounds; this ignores
  cross-month correlation, so the range is an approximation (usually wider).
- The models capture historical trend and seasonality only — **not** fare
  policy, new routes, or external shocks. Treat the output as an indicative
  estimate, not a certainty.
- Source data: BPS. `Total` and `Jawa (...)` are aggregates; never sum them with
  their components.